# Step 8 — Biomarkers and Synthesis (α–β–γ Strategy Analysis)

**Purpose:** Extract per-sequence maturation strategy vectors (α, β, γ) from the three objectives and characterize strategy distributions by germline, isotype, and donor. Compile germline fingerprints.

**Strategy vector definition:**
```
α(x) = Φ_S(x) / [Φ_S(x) + Φ_A(x) + Φ_R(x)]   ← structural cost share
β(x) = Φ_A(x) / [Φ_S(x) + Φ_A(x) + Φ_R(x)]   ← affinity cost share
γ(x) = Φ_R(x) / [Φ_S(x) + Φ_A(x) + Φ_R(x)]   ← reactivity cost share
α + β + γ = 1
```

**Calculations:**
- B1: Strategy simplex — all memory sequences in (α, β, γ) ternary space
- B2: Per-germline profiles — mean strategy + confidence ellipses on ternary
- B3: Per-isotype profiles — isotype-level shifts; Kruskal-Wallis test
- B4: Germline fingerprint — composite table (R_AID, λ_S, λ_R, mean strategy, Pareto HV)

**Inputs:**
- `results/tables/affinity_proxy.parquet` — Φ_A, mutation counts
- `results/tables/phi_r_scores.parquet` — Φ_R per sequence
- `results/tables/omega_per_position.parquet` — ω per position for Φ_S calibration
- `results/tables/lambda_by_germline.csv` — λ_S, λ_R per germline (Step 5)
- `results/tables/lambda_by_germline_endpoints.csv` — λ endpoint-level (Step 5b)
- `results/tables/evolvability_index.csv` — R_AID per germline (Step 1)
- `results/tables/pareto_by_germline.csv` — Pareto hypervolume per germline (Step 6)

**Outputs:**
- `results/tables/strategy_vectors.parquet` + `.csv`
- `results/tables/strategy_by_germline.csv`
- `results/tables/strategy_by_isotype.csv`
- `results/tables/germline_fingerprints.csv`
- `results/figures/fig_b1_strategy_ternary.png` + `.csv`
- `results/figures/fig_b2_germline_strategy.png` + `.csv`
- `results/figures/fig_b3_isotype_strategy.png` + `.csv`
- `results/figures/fig_b4_fingerprints.png` + `.csv`

In [ ]:
import polars as pl
import numpy as np
import math
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.patches import Ellipse
from matplotlib.lines import Line2D

from scipy.stats import kruskal, mannwhitneyu
from scipy.optimize import nnls

In [ ]:
DATA_DIR = Path("/home/jovyan/shared/Benjamin/LineageAtlas/pairplex_paper/")
RESULTS  = DATA_DIR / "results"
FIGURES  = RESULTS / "figures"
TABLES   = RESULTS / "tables"

# Thresholds
MIN_GERM_N  = 100   # minimum sequences for per-germline strategy profile
MIN_ISO_N   = 1000  # minimum sequences for per-isotype analysis
MIN_DONOR_N = 500   # minimum sequences for per-donor analysis

# Ternary plot constants
SQRT3_2 = math.sqrt(3) / 2

In [ ]:
# Calibrate PHI_S_CDR and PHI_S_FWR from omega_per_position.parquet
# (same as Step 7 cell 03)
omega_df = pl.read_parquet(TABLES / "omega_per_position.parquet")

# Use S5F-weighted mean -log(omega) per region, same as Step 7
omega_df = omega_df.with_columns(
    pl.when(pl.col('omega') > 0)
    .then(-pl.col('omega').log())
    .otherwise(None)
    .alias('neg_log_omega')
)

region_means = (
    omega_df
    .filter(pl.col('neg_log_omega').is_not_null())
    .with_columns(
        pl.when(pl.col('region').str.contains('CDR')).then(pl.lit('CDR'))
        .otherwise(pl.lit('FWR')).alias('region_class')
    )
    .group_by('region_class')
    .agg(pl.col('neg_log_omega').mean().alias('mean_neg_log_omega'))
)

PHI_S_CDR = region_means.filter(pl.col('region_class') == 'CDR')['mean_neg_log_omega'][0]
PHI_S_FWR = region_means.filter(pl.col('region_class') == 'FWR')['mean_neg_log_omega'][0]

print(f"PHI_S_CDR = {PHI_S_CDR:.4f}")
print(f"PHI_S_FWR = {PHI_S_FWR:.4f}")

In [ ]:
# Load phi_A (with mutation counts) and phi_R
phi_a_df = pl.read_parquet(TABLES / "affinity_proxy.parquet")
phi_r_df = pl.read_parquet(TABLES / "phi_r_scores.parquet")

data = (
    phi_a_df
    .filter(pl.col('phi_A').is_not_null())
    .select([
        'seq_name', 'v_gene:0', 'isotype_class', 'donor', 'lineage',
        'n_R_CDR_H', 'n_S_CDR_H', 'n_R_FWR_H', 'n_S_FWR_H', 'n_mut_H', 'phi_A'
    ])
    .join(phi_r_df.select(['seq_name', 'phi_R']), on='seq_name', how='inner')
    .with_columns([
        (pl.col('n_R_CDR_H') + pl.col('n_S_CDR_H')).alias('n_mut_CDR_H'),
        (pl.col('n_R_FWR_H') + pl.col('n_S_FWR_H')).alias('n_mut_FWR_H'),
    ])
    .with_columns(
        (pl.col('n_mut_CDR_H') * PHI_S_CDR
         + pl.col('n_mut_FWR_H') * PHI_S_FWR).alias('phi_S')
    )
    .filter(pl.col('phi_A').is_not_null())
    .filter(pl.col('phi_R').is_not_null())
)

print(f"Full dataset: {data.height:,} sequences")
print(f"  phi_A range: [{data['phi_A'].min():.3f}, {data['phi_A'].max():.3f}]")
print(f"  phi_S range: [{data['phi_S'].min():.3f}, {data['phi_S'].max():.3f}]")
print(f"  phi_R range: [{data['phi_R'].min():.3f}, {data['phi_R'].max():.3f}]")

In [ ]:
# Compute strategy vectors alpha, beta, gamma
# Requires all three objectives to be > 0 and total > epsilon
EPS = 1e-6

data = (
    data
    .with_columns([
        # Clamp phi_A >= 0 (R/S ratio should be non-negative; protect against float noise)
        pl.col('phi_A').clip(lower_bound=0.0).alias('phi_A'),
        pl.col('phi_S').clip(lower_bound=0.0).alias('phi_S'),
        pl.col('phi_R').clip(lower_bound=0.0).alias('phi_R'),
    ])
    .with_columns(
        (pl.col('phi_S') + pl.col('phi_A') + pl.col('phi_R')).alias('phi_total')
    )
    .filter(pl.col('phi_total') > EPS)  # exclude unmutated or degenerate sequences
    .with_columns([
        (pl.col('phi_S') / pl.col('phi_total')).alias('alpha'),   # structural share
        (pl.col('phi_A') / pl.col('phi_total')).alias('beta'),    # affinity share
        (pl.col('phi_R') / pl.col('phi_total')).alias('gamma'),   # reactivity share
    ])
)

print(f"Sequences with valid strategy vector: {data.height:,}")
print(f"  Mean alpha (structural): {data['alpha'].mean():.3f}")
print(f"  Mean beta  (affinity):   {data['beta'].mean():.3f}")
print(f"  Mean gamma (reactivity): {data['gamma'].mean():.3f}")
print(f"  Check sum = 1: {(data['alpha'] + data['beta'] + data['gamma']).mean():.6f}")

In [ ]:
# Helper functions for ternary plots

def ternary_to_cartesian(alpha, beta, gamma):
    """Convert (alpha, beta, gamma) to Cartesian (x, y).
    Vertices:
      alpha (structural) → top:          (0.5, sqrt(3)/2)
      beta  (affinity)   → bottom-left:  (0, 0)
      gamma (reactivity) → bottom-right: (1, 0)
    """
    x = 0.5 * alpha + gamma
    y = SQRT3_2 * alpha
    return x, y


def draw_ternary_frame(ax, labels=('α (Structural)', 'β (Affinity)', 'γ (Reactivity)'),
                       fontsize=10, gridlines=True):
    """Draw equilateral triangle frame for ternary plot."""
    triangle = plt.Polygon(
        [[0, 0], [1, 0], [0.5, SQRT3_2]],
        fill=False, edgecolor='black', lw=1.5
    )
    ax.add_patch(triangle)

    # Vertex labels
    ax.text(0.5, SQRT3_2 + 0.03, labels[0], ha='center', va='bottom', fontsize=fontsize, fontweight='bold')
    ax.text(-0.08, -0.04, labels[1], ha='center', va='top', fontsize=fontsize, fontweight='bold')
    ax.text(1.08, -0.04, labels[2], ha='center', va='top', fontsize=fontsize, fontweight='bold')

    if gridlines:
        for frac in [0.25, 0.5, 0.75]:
            # Lines of constant alpha
            x1, y1 = ternary_to_cartesian(frac, 1-frac, 0)
            x2, y2 = ternary_to_cartesian(frac, 0, 1-frac)
            ax.plot([x1, x2], [y1, y2], 'grey', lw=0.5, alpha=0.5, ls='--')
            # Lines of constant beta
            x1, y1 = ternary_to_cartesian(1-frac, frac, 0)
            x2, y2 = ternary_to_cartesian(0, frac, 1-frac)
            ax.plot([x1, x2], [y1, y2], 'grey', lw=0.5, alpha=0.5, ls='--')
            # Lines of constant gamma
            x1, y1 = ternary_to_cartesian(1-frac, 0, frac)
            x2, y2 = ternary_to_cartesian(0, 1-frac, frac)
            ax.plot([x1, x2], [y1, y2], 'grey', lw=0.5, alpha=0.5, ls='--')

    ax.set_xlim(-0.15, 1.15)
    ax.set_ylim(-0.10, SQRT3_2 + 0.10)
    ax.set_aspect('equal')
    ax.axis('off')


def draw_covariance_ellipse(ax, x_mean, y_mean, cov_xy, n_std=1.5, color='black', alpha=0.6, lw=1.5):
    """Draw a covariance ellipse in Cartesian ternary space."""
    if np.any(np.isnan(cov_xy)):
        return
    try:
        eigenvalues, eigenvectors = np.linalg.eigh(cov_xy)
        eigenvalues = np.maximum(eigenvalues, 0)  # clip numerical negatives
        width  = 2 * n_std * np.sqrt(eigenvalues[1])
        height = 2 * n_std * np.sqrt(eigenvalues[0])
        angle  = np.degrees(np.arctan2(eigenvectors[1, 1], eigenvectors[0, 1]))
        ellipse = Ellipse(
            xy=(x_mean, y_mean), width=width, height=height, angle=angle,
            edgecolor=color, facecolor='none', lw=lw, alpha=alpha
        )
        ax.add_patch(ellipse)
    except np.linalg.LinAlgError:
        pass

## B1 — Strategy Simplex: All Memory Sequences

**Question:** Where do memory sequences cluster on the (α, β, γ) ternary simplex?  
**Expected:** Sequences should concentrate in a region consistent with the trade-off between structural constraint (high α) and affinity gain (high β). Reactivity-dominated sequences (high γ) should be rare (checkpoint removes autoreactive cells).

In [ ]:
# Convert to Cartesian ternary coordinates
alpha_arr = data['alpha'].to_numpy()
beta_arr  = data['beta'].to_numpy()
gamma_arr = data['gamma'].to_numpy()

x_cart, y_cart = ternary_to_cartesian(alpha_arr, beta_arr, gamma_arr)

# Isotype color mapping
ISO_COLORS = {'IgM': '#1f77b4', 'IgG': '#d62728', 'IgA': '#2ca02c', 'IgE': '#ff7f0e'}

isotype_arr = data['isotype_class'].to_list()
colors = [ISO_COLORS.get(iso, '#7f7f7f') for iso in isotype_arr]

print(f"Ternary coordinates computed for {len(x_cart):,} sequences")
print(f"  x range: [{x_cart.min():.3f}, {x_cart.max():.3f}]")
print(f"  y range: [{y_cart.min():.3f}, {y_cart.max():.3f}]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('B1 — Strategy Simplex: Memory Sequences', fontsize=13, fontweight='bold', y=1.01)

# --- Panel A: density hexbin (all sequences) ---
ax = axes[0]
draw_ternary_frame(ax)
hb = ax.hexbin(x_cart, y_cart, gridsize=60, cmap='YlOrRd', mincnt=1, bins='log')
plt.colorbar(hb, ax=ax, label='log10(count)')
ax.set_title('Density (all memory sequences)', fontsize=11)

# --- Panel B: isotype colored (subsample for visibility) ---
ax = axes[1]
draw_ternary_frame(ax)
rng = np.random.default_rng(42)
idx_sub = rng.choice(len(x_cart), size=min(50_000, len(x_cart)), replace=False)
for iso, col in ISO_COLORS.items():
    mask = np.array([isotype_arr[i] == iso for i in idx_sub])
    if mask.sum() > 0:
        ax.scatter(x_cart[idx_sub][mask], y_cart[idx_sub][mask],
                   c=col, s=2, alpha=0.3, label=iso, rasterized=True)
ax.legend(loc='upper right', fontsize=8, markerscale=4)
ax.set_title('Isotype distribution (50K subsample)', fontsize=11)

plt.tight_layout()
plt.savefig(FIGURES / "fig_b1_strategy_ternary.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_b1_strategy_ternary.png")

In [ ]:
# Save strategy vectors
strategy_df = data.select([
    'seq_name', 'v_gene:0', 'isotype_class', 'donor', 'lineage',
    'phi_S', 'phi_A', 'phi_R', 'phi_total', 'alpha', 'beta', 'gamma', 'n_mut_H'
])
strategy_df.write_parquet(TABLES / "strategy_vectors.parquet")
strategy_df.write_csv(TABLES / "strategy_vectors.csv")

# Save figure data (isotype means on ternary)
iso_means = (
    data
    .group_by('isotype_class')
    .agg([
        pl.col('alpha').mean().alias('mean_alpha'),
        pl.col('beta').mean().alias('mean_beta'),
        pl.col('gamma').mean().alias('mean_gamma'),
        pl.col('alpha').std().alias('std_alpha'),
        pl.col('beta').std().alias('std_beta'),
        pl.col('gamma').std().alias('std_gamma'),
        pl.len().alias('n'),
    ])
    .sort('n', descending=True)
)
iso_means.write_csv(FIGURES / "fig_b1_strategy_ternary.csv")
print(iso_means)

## B2 — Per-Germline Strategy Profiles

**Question:** Do different IGHV germlines occupy distinct regions of the strategy simplex?  
**Expected:** bnAb germlines (IGHV1-2, IGHV1-69) may cluster in high-β (affinity-driven) or high-γ (reactivity-constrained) regions. Germlines with high λ_R from Step 5 may cluster in high-γ region.

In [ ]:
# Per-germline mean strategy + covariance in Cartesian ternary space
germline_results = []

for key, sub in data.filter(pl.col('v_gene:0').is_not_null()).partition_by('v_gene:0', as_dict=True).items():
    vgene = key[0] if isinstance(key, (list, tuple)) else str(key)
    n = sub.height
    if n < MIN_GERM_N:
        continue

    a = sub['alpha'].to_numpy()
    b = sub['beta'].to_numpy()
    g = sub['gamma'].to_numpy()
    x, y = ternary_to_cartesian(a, b, g)

    x_mean, y_mean = x.mean(), y.mean()
    cov_xy = np.cov(np.vstack([x, y]))

    germline_results.append({
        'v_gene': vgene,
        'n': n,
        'mean_alpha': float(a.mean()),
        'mean_beta':  float(b.mean()),
        'mean_gamma': float(g.mean()),
        'std_alpha': float(a.std()),
        'std_beta':  float(b.std()),
        'std_gamma': float(g.std()),
        'x_mean': float(x_mean),
        'y_mean': float(y_mean),
        'cov_xx': float(cov_xy[0, 0]),
        'cov_xy': float(cov_xy[0, 1]),
        'cov_yy': float(cov_xy[1, 1]),
    })

germ_df = pl.DataFrame(germline_results).sort('n', descending=True)
print(f"Germlines with n >= {MIN_GERM_N}: {germ_df.height}")
print(germ_df.head(10))

In [ ]:
# Ternary plot with per-germline ellipses; highlight bnAb germlines
BNAB_GERMLINES = {'IGHV1-2', 'IGHV1-69', 'IGHV3-30', 'IGHV4-34'}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('B2 — Per-Germline Strategy Profiles', fontsize=13, fontweight='bold', y=1.01)

germ_arr = germ_df.to_dicts()
n_vals = np.array([g['n'] for g in germ_arr])
norm = mcolors.LogNorm(vmin=n_vals.min(), vmax=n_vals.max())
cmap = plt.cm.viridis

# --- Panel A: mean positions colored by n ---
ax = axes[0]
draw_ternary_frame(ax)
sc = ax.scatter(
    [g['x_mean'] for g in germ_arr],
    [g['y_mean'] for g in germ_arr],
    c=[g['n'] for g in germ_arr],
    norm=norm, cmap=cmap, s=60, zorder=3, edgecolors='white', lw=0.5
)
plt.colorbar(sc, ax=ax, label='n sequences')
# Label bnAb germlines
for g in germ_arr:
    if g['v_gene'] in BNAB_GERMLINES:
        ax.annotate(g['v_gene'], (g['x_mean'], g['y_mean']),
                    fontsize=7, ha='left', va='bottom',
                    xytext=(4, 4), textcoords='offset points',
                    arrowprops=dict(arrowstyle='-', lw=0.5))
ax.set_title('Mean strategy per germline (size ∝ n)', fontsize=11)

# --- Panel B: top-20 germlines with confidence ellipses ---
ax = axes[1]
draw_ternary_frame(ax)

top20 = germ_arr[:20]
colors_20 = plt.cm.tab20(np.linspace(0, 1, 20))
legend_handles = []
for i, g in enumerate(top20):
    col = colors_20[i]
    cov_mat = np.array([[g['cov_xx'], g['cov_xy']], [g['cov_xy'], g['cov_yy']]])
    draw_covariance_ellipse(ax, g['x_mean'], g['y_mean'], cov_mat, n_std=1.5, color=col)
    ax.scatter(g['x_mean'], g['y_mean'], c=[col], s=40, zorder=4, edgecolors='white', lw=0.5)
    legend_handles.append(mpatches.Patch(color=col, label=f"{g['v_gene']} (n={g['n']:,})"))

ax.legend(handles=legend_handles, loc='upper left', fontsize=6,
          ncol=2, bbox_to_anchor=(-0.05, 1.0))
ax.set_title('Top-20 germlines: 1.5σ confidence ellipses', fontsize=11)

plt.tight_layout()
plt.savefig(FIGURES / "fig_b2_germline_strategy.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_b2_germline_strategy.png")

In [ ]:
strategy_by_germline = germ_df.select([
    'v_gene', 'n', 'mean_alpha', 'mean_beta', 'mean_gamma',
    'std_alpha', 'std_beta', 'std_gamma'
])
strategy_by_germline.write_csv(TABLES / "strategy_by_germline.csv")
germ_df.write_csv(FIGURES / "fig_b2_germline_strategy.csv")
print(f"Saved strategy_by_germline.csv ({strategy_by_germline.height} germlines)")

## B3 — Per-Isotype Strategy Profiles

**Question:** Does isotype class switching shift the maturation strategy?  
**Expected:**
- IgG should show highest β (affinity-dominated) — deepest GC selection
- IgM should show higher α (structural cost dominant) — less affinity selection
- IgA intermediate due to T-independent switching dilution  
**Test:** Kruskal-Wallis on α, β, γ across isotypes; pairwise Mann-Whitney U for significant cases.

In [ ]:
isotypes_order = ['IgM', 'IgG', 'IgA', 'IgE']
iso_data = {
    iso: data.filter((pl.col('isotype_class') == iso) & (pl.col('isotype_class').is_not_null()))
    for iso in isotypes_order
}

iso_stats = []
for iso, sub in iso_data.items():
    if sub.height < MIN_ISO_N:
        continue
    iso_stats.append({
        'isotype': iso,
        'n': sub.height,
        'mean_alpha': float(sub['alpha'].mean()),
        'mean_beta':  float(sub['beta'].mean()),
        'mean_gamma': float(sub['gamma'].mean()),
        'median_alpha': float(sub['alpha'].median()),
        'median_beta':  float(sub['beta'].median()),
        'median_gamma': float(sub['gamma'].median()),
        'std_alpha': float(sub['alpha'].std()),
        'std_beta':  float(sub['beta'].std()),
        'std_gamma': float(sub['gamma'].std()),
    })

iso_df = pl.DataFrame(iso_stats)
print(iso_df)

# Kruskal-Wallis tests
valid_isos = [iso for iso in isotypes_order if iso_data[iso].height >= MIN_ISO_N]
groups_alpha = [iso_data[iso]['alpha'].to_numpy() for iso in valid_isos]
groups_beta  = [iso_data[iso]['beta'].to_numpy() for iso in valid_isos]
groups_gamma = [iso_data[iso]['gamma'].to_numpy() for iso in valid_isos]

kw_alpha = kruskal(*groups_alpha)
kw_beta  = kruskal(*groups_beta)
kw_gamma = kruskal(*groups_gamma)

print(f"\nKruskal-Wallis tests across isotypes ({', '.join(valid_isos)}):")
print(f"  alpha: H={kw_alpha.statistic:.2f}, p={kw_alpha.pvalue:.2e}")
print(f"  beta:  H={kw_beta.statistic:.2f},  p={kw_beta.pvalue:.2e}")
print(f"  gamma: H={kw_gamma.statistic:.2f}, p={kw_gamma.pvalue:.2e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('B3 — Per-Isotype Strategy Profiles', fontsize=13, fontweight='bold', y=1.01)

strategy_labels = ['α (Structural)', 'β (Affinity)', 'γ (Reactivity)']
strategy_cols   = ['alpha', 'beta', 'gamma']
kw_stats = [kw_alpha, kw_beta, kw_gamma]
iso_colors = [ISO_COLORS.get(iso, '#7f7f7f') for iso in valid_isos]

for ax, col, label, kw in zip(axes, strategy_cols, strategy_labels, kw_stats):
    bp_data = [iso_data[iso][col].to_numpy() for iso in valid_isos]
    bp = ax.boxplot(bp_data, labels=valid_isos, patch_artist=True, showfliers=False,
                    medianprops=dict(color='black', lw=2))
    for patch, col_val in zip(bp['boxes'], iso_colors):
        patch.set_facecolor(col_val)
        patch.set_alpha(0.7)
    ax.set_title(f'{label}\nKW: H={kw.statistic:.1f}, p={kw.pvalue:.1e}', fontsize=10)
    ax.set_ylabel(label, fontsize=9)
    ax.set_xlabel('Isotype', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / "fig_b3_isotype_strategy.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_b3_isotype_strategy.png")

In [ ]:
iso_df.write_csv(TABLES / "strategy_by_isotype.csv")
iso_df.write_csv(FIGURES / "fig_b3_isotype_strategy.csv")
print(f"Saved strategy_by_isotype.csv ({iso_df.height} isotypes)")

## B4 — Germline Fingerprints

**What:** Compile a composite per-germline summary table linking evolvability (R_AID), constraint prices (λ_S, λ_R from Steps 5 and 5b), mean strategy (α, β, γ), and Pareto hypervolume.  
**Biological question:** Do germlines with high reactivity constraint (high λ_R) cluster in a specific strategy region? Do bnAb germlines show distinct fingerprints?

In [ ]:
# Load per-germline metrics from all prior steps
lambda_germ    = pl.read_csv(TABLES / "lambda_by_germline.csv")
lambda_ep      = pl.read_csv(TABLES / "lambda_by_germline_endpoints.csv")
evolvability   = pl.read_csv(TABLES / "evolvability_index.csv")
pareto_germ    = pl.read_csv(TABLES / "pareto_by_germline.csv")
strategy_germ  = pl.read_csv(TABLES / "strategy_by_germline.csv")

# Rename endpoint lambda cols to avoid collision
lambda_ep = lambda_ep.rename({
    'lambda_S': 'lambda_S_ep', 'lambda_R': 'lambda_R_ep',
    'r2': 'r2_ep', 'n': 'n_ep'
})

# Build fingerprint table
fingerprints = (
    strategy_germ
    .join(lambda_germ.select(['v_gene', 'lambda_S', 'lambda_R', 'r2', 'n']),
          on='v_gene', how='left')
    .join(lambda_ep.select(['v_gene', 'lambda_S_ep', 'lambda_R_ep', 'r2_ep', 'n_ep']),
          on='v_gene', how='left')
    .join(evolvability.select(['v_gene', 'R_AID', 'density_cdr', 'density_fwr']),
          on='v_gene', how='left')
    .join(pareto_germ.select(['v_gene', 'n_front', 'pct_front', 'hv2d_proxy',
                               'mean_phi_S', 'mean_phi_A', 'mean_phi_R']),
          on='v_gene', how='left')
    .sort('n', descending=True)
)

print(f"Germline fingerprints: {fingerprints.height} germlines")
print(fingerprints.select([
    'v_gene', 'n', 'mean_alpha', 'mean_beta', 'mean_gamma',
    'lambda_S', 'lambda_R', 'R_AID', 'hv2d_proxy'
]).head(10))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('B4 — Germline Fingerprints', fontsize=13, fontweight='bold', y=1.01)

fp = fingerprints.to_pandas()
fp_large = fp[fp['n'] >= MIN_GERM_N].dropna(subset=['lambda_R', 'mean_gamma', 'R_AID', 'hv2d_proxy'])

# Panel A: λ_R (Step 5) vs mean γ (reactivity share)
ax = axes[0, 0]
ax.scatter(fp_large['lambda_R'], fp_large['mean_gamma'],
           s=np.log1p(fp_large['n']) * 5, alpha=0.7, edgecolors='white', lw=0.5, c='steelblue')
for _, row in fp_large[fp_large['v_gene'].isin(BNAB_GERMLINES)].iterrows():
    ax.annotate(row['v_gene'], (row['lambda_R'], row['mean_gamma']), fontsize=7,
                xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('λ_R (Step 5 cross-sectional)', fontsize=10)
ax.set_ylabel('Mean γ (reactivity share)', fontsize=10)
ax.set_title('Reactivity constraint vs strategy', fontsize=11)

# Panel B: R_AID vs mean β (affinity share)
ax = axes[0, 1]
ax.scatter(fp_large['R_AID'], fp_large['mean_beta'],
           s=np.log1p(fp_large['n']) * 5, alpha=0.7, edgecolors='white', lw=0.5, c='seagreen')
for _, row in fp_large[fp_large['v_gene'].isin(BNAB_GERMLINES)].iterrows():
    ax.annotate(row['v_gene'], (row['R_AID'], row['mean_beta']), fontsize=7,
                xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('R_AID (evolvability)', fontsize=10)
ax.set_ylabel('Mean β (affinity share)', fontsize=10)
ax.set_title('Evolvability vs affinity strategy', fontsize=11)

# Panel C: Pareto hypervolume vs mean β
ax = axes[1, 0]
ax.scatter(fp_large['hv2d_proxy'], fp_large['mean_beta'],
           s=np.log1p(fp_large['n']) * 5, alpha=0.7, edgecolors='white', lw=0.5, c='darkorange')
for _, row in fp_large[fp_large['v_gene'].isin(BNAB_GERMLINES)].iterrows():
    ax.annotate(row['v_gene'], (row['hv2d_proxy'], row['mean_beta']), fontsize=7,
                xytext=(4, 4), textcoords='offset points')
ax.set_xlabel('Pareto hypervolume proxy', fontsize=10)
ax.set_ylabel('Mean β (affinity share)', fontsize=10)
ax.set_title('Pareto frontier breadth vs affinity strategy', fontsize=11)

# Panel D: ternary ternary of germline means colored by lambda_R
ax = axes[1, 1]
draw_ternary_frame(ax)
sc = ax.scatter(
    fp_large.apply(lambda r: ternary_to_cartesian(r['mean_alpha'], r['mean_beta'], r['mean_gamma'])[0], axis=1),
    fp_large.apply(lambda r: ternary_to_cartesian(r['mean_alpha'], r['mean_beta'], r['mean_gamma'])[1], axis=1),
    c=fp_large['lambda_R'], cmap='Reds',
    s=np.log1p(fp_large['n']) * 6, alpha=0.8, edgecolors='white', lw=0.5
)
plt.colorbar(sc, ax=ax, label='λ_R (Step 5)')
for _, row in fp_large[fp_large['v_gene'].isin(BNAB_GERMLINES)].iterrows():
    xc, yc = ternary_to_cartesian(row['mean_alpha'], row['mean_beta'], row['mean_gamma'])
    ax.annotate(row['v_gene'], (xc, yc), fontsize=7, xytext=(4, 4), textcoords='offset points')
ax.set_title('Germline strategy colored by λ_R', fontsize=11)

plt.tight_layout()
plt.savefig(FIGURES / "fig_b4_fingerprints.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved fig_b4_fingerprints.png")

In [ ]:
fingerprints.write_csv(TABLES / "germline_fingerprints.csv")
fingerprints.write_csv(FIGURES / "fig_b4_fingerprints.csv")
print(f"Saved germline_fingerprints.csv ({fingerprints.height} germlines)")
print("\nTop 10 by λ_R:")
print(fingerprints.sort('lambda_R', descending=True, nulls_last=True)
      .select(['v_gene', 'n', 'lambda_R', 'mean_alpha', 'mean_beta', 'mean_gamma', 'R_AID', 'hv2d_proxy'])
      .head(10))

## Summary — Step 8

### B1 — Strategy simplex
All memory sequences projected onto (α, β, γ) ternary. Expected: concentration in high-β (affinity-driven) and moderate-α region, consistent with SHM selecting for affinity while accepting structural cost.

### B2 — Germline strategy profiles
Per-germline mean strategy + 1.5σ confidence ellipses. Key question: do bnAb germlines (IGHV1-2, IGHV1-69) cluster in high-γ (reactivity-constrained) region, consistent with their elevated λ_R from Step 5?

### B3 — Isotype strategy profiles
Kruskal-Wallis test on α, β, γ across isotypes. Expected: IgG shows highest β (deepest affinity selection); IgM shows highest α (structural cost dominant at lower maturation depth).

### B4 — Germline fingerprints
Composite table: R_AID (evolvability), λ_S/λ_R (constraint prices), mean strategy, Pareto HV. Key cross-step concordance check: germlines with high λ_R from Step 5 should cluster in high-γ region on the ternary.

### Outputs
| File | Description |
|------|-------------|
| `strategy_vectors.parquet` | Per-sequence α, β, γ |
| `strategy_by_germline.csv` | Per-germline mean ± std of α, β, γ |
| `strategy_by_isotype.csv` | Per-isotype means + KW statistics |
| `germline_fingerprints.csv` | Composite R_AID, λ, strategy, Pareto HV per germline |

### Next steps
- Step 5c (`05c_lagrangian.ipynb`): germline-as-lineage endpoint approach to recover λ_R signal from all ~1.5M memory sequences
- Per-isotype stratified D2/D3 analysis in Step 7 (IgG-only trajectories)
- Fix Φ_S monotonicity for within-lineage KKT (Step 7b)